# Evaluation of the nulled field

Created on 26. Aug. 2026

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 

import External_functions as fkt                                        # longer plots, file readouts and other larger functions are stored in this file to save space

In [ ]:
# ---------- Matplotlib-Einstellungen ----------
import matplotlib as mpl

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "pgf.rcfonts": True,  # let LaTeX control fonts

    # Do NOT specify "Computer Modern Roman" here; let LaTeX handle it.
    # Optionally just say "serif" or "sans-serif" as a hint:
    "font.family": "serif",  # or "sans-serif" if your thesis is sans

    "pgf.preamble": "\n".join([
        r"\usepackage{amsmath}",
        r"\usepackage[T1]{fontenc}",
    ]),

    "font.size": 10,
    "axes.labelsize": "medium",
    "axes.titlesize": "medium",
    "xtick.labelsize": "small",
    "ytick.labelsize": "small",
    "legend.fontsize": "small",
})


In [ ]:
# Specify folders containing the residual-field point data
residual_field_tobi_folder = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Finale_Karten_Tobi\Averaged_B_field"
residual_field_gregor_folder = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Nulling_fields\Residuals_nuliing_AGAIN_2026-09-03_13-18-07\map\points"

# If the .npz file does not contain geometric track data, specify the coordinate shifts here
coordinate_shift_x = 0.400                              # shift of the mapped volume in the x-direction
coordinate_shift_y = 0.400                              # shift of the mapped volume in the y-direction
coordinate_shift_z = 0.400                              # shift of the mapped volume in the z-direction  

# Load the data using the load_data_from_folder function from External_functions.py.
residual_field_coordinates, B_residual_field_tobi, B_residual_field_gregor = fkt.load_data_from_folder(
    residual_field_tobi_folder,
    residual_field_gregor_folder,
    coordinate_shift_x,
    coordinate_shift_y,
    coordinate_shift_z,
)

# Transform the magnetic-field components from the QSpin to the MSR coordinate system
apply_qspin_to_msr_transform = True
tobi_field_offsets = [-100.16, -4.67, 74.58]# [0.0, 0.0, 0.0]
gregor_field_offsets = [0, 0, 0]#[-74.44e-12, 29.63e-12, 51.49e-12]
remove_field_offset = False
if apply_qspin_to_msr_transform:
    Bx = B_residual_field_tobi[:, 0].copy()
    By = B_residual_field_tobi[:, 1].copy()
    Bz = B_residual_field_tobi[:, 2].copy()

    if remove_field_offset:
        Bx = Bx - tobi_field_offsets[0]
        By = By - tobi_field_offsets[1]
        Bz = Bz - tobi_field_offsets[2]

    B_residual_field_tobi[:, 0] = Bx
    B_residual_field_tobi[:, 1] = -Bz
    B_residual_field_tobi[:, 2] = -By

    Bx = B_residual_field_gregor[:, 0].copy()
    By = B_residual_field_gregor[:, 1].copy()
    Bz = B_residual_field_gregor[:, 2].copy()

    if remove_field_offset:
        Bx = Bx - gregor_field_offsets[0]
        By = By - gregor_field_offsets[1]
        Bz = Bz - gregor_field_offsets[2]

    B_residual_field_gregor[:, 0] = Bx
    B_residual_field_gregor[:, 1] = -Bz
    B_residual_field_gregor[:, 2] = -By
    print('Changed the magnetic-field directions from the left-handed QSpin coordinate system to the right-handed MSR coordinate system to match the spatial coordinates.')

In [ ]:
B_difference_field = B_residual_field_gregor - B_residual_field_tobi

# Plot the three residual-field datasets in 3D
fig = plt.figure(figsize=(18, 5))

fields = [
    (B_residual_field_tobi, r'$|\mathbf{B}_{\mathrm{Tobi}}|$', 'Residual Field (Tobi)', residual_field_coordinates),
    (B_residual_field_gregor, r'$|\mathbf{B}_{\mathrm{Gregor}}|$', 'Residual Field (Gregor)', residual_field_coordinates),
    (B_difference_field, r'$|\mathbf{B}_{\mathrm{difference}}|$', 'Difference Field (Gregor - Tobi)', residual_field_coordinates),
]

for plot_index, (B_field, colorbar_label, title, plot_coordinates) in enumerate(fields):
    ax = fig.add_subplot(1, 3, plot_index + 1, projection='3d')

    field_magnitude = np.linalg.norm(B_field, axis=1)

    # Formatting: slightly reduce the scatter-marker size.
    scatter = ax.scatter(
        plot_coordinates[:, 0],
        plot_coordinates[:, 1],
        plot_coordinates[:, 2],
        c=field_magnitude,
        s=70,
        cmap='viridis',
        alpha=0.8
    )

    ax.set_title(title, pad=10)
    # Formatting: shorten and separate the 3D colourbar from the z-axis label.
    fig.colorbar(scatter, ax=ax, label=colorbar_label, pad=0.15, shrink=0.5)
    ax.set_xlabel(r'$x$ [m]')
    ax.set_ylabel(r'$y$ [m]')
    ax.set_zlabel(r'$z$ [m]')
    ax.view_init(elev=20, azim=-60)

fig.tight_layout()

# Extract the points and field vectors on each symmetry plane
def filter_symmetry_plane(coordinates, B_field, plane_coordinate, tolerance=1e-5):
    """Return the coordinates and field vectors located on a symmetry plane."""
    plane_mask = np.isclose(coordinates[:, plane_coordinate], 0, atol=tolerance)
    return coordinates[plane_mask], B_field[plane_mask], plane_mask

# Filter the measured fields for each symmetry plane
coordinates_X0, B_tobi_X0, _ = filter_symmetry_plane(residual_field_coordinates, B_residual_field_tobi, 0)
coordinates_Y0, B_tobi_Y0, _ = filter_symmetry_plane(residual_field_coordinates, B_residual_field_tobi, 1)
coordinates_Z0, B_tobi_Z0, _ = filter_symmetry_plane(residual_field_coordinates, B_residual_field_tobi, 2)

coordinates_X0_gregor, B_gregor_X0, _ = filter_symmetry_plane(residual_field_coordinates, B_residual_field_gregor, 0)
coordinates_Y0_gregor, B_gregor_Y0, _ = filter_symmetry_plane(residual_field_coordinates, B_residual_field_gregor, 1)
coordinates_Z0_gregor, B_gregor_Z0, _ = filter_symmetry_plane(residual_field_coordinates, B_residual_field_gregor, 2)

coordinates_X0_difference, B_difference_X0, _ = filter_symmetry_plane(residual_field_coordinates, B_difference_field, 0)
coordinates_Y0_difference, B_difference_Y0, _ = filter_symmetry_plane(residual_field_coordinates, B_difference_field, 1)
coordinates_Z0_difference, B_difference_Z0, _ = filter_symmetry_plane(residual_field_coordinates, B_difference_field, 2)

# Create a 3x3 grid of vector plots for the three fields and three symmetry planes
fig = plt.figure(figsize=(18, 12))

fields = [
    (B_residual_field_tobi, coordinates_X0, B_tobi_X0, coordinates_Y0, B_tobi_Y0, coordinates_Z0, B_tobi_Z0,
     r'$|\mathbf{B}_{\mathrm{Tobi}}|$', 'Residual Field (Tobi)', residual_field_coordinates),
    (B_residual_field_gregor, coordinates_X0_gregor, B_gregor_X0, coordinates_Y0_gregor, B_gregor_Y0, coordinates_Z0_gregor, B_gregor_Z0,
     r'$|\mathbf{B}_{\mathrm{Gregor}}|$', 'Residual Field (Gregor)', residual_field_coordinates),
    (B_difference_field, coordinates_X0_difference, B_difference_X0, coordinates_Y0_difference, B_difference_Y0, coordinates_Z0_difference, B_difference_Z0,
     r'$|\mathbf{B}_{\mathrm{difference}}|$', 'Difference Field (Gregor - Tobi)', residual_field_coordinates),
]

symmetry_planes = [
    (r'$x$ = 0', 1, 2, r'$y$ [m]', r'$z$ [m]'),
    (r'$y$ = 0', 0, 2, r'$x$ [m]', r'$z$ [m]'),
    (r'$z$ = 0', 0, 1, r'$x$ [m]', r'$y$ [m]')
]

for row_index, (plane_name, coordinate_1, coordinate_2, axis_1_label, axis_2_label) in enumerate(symmetry_planes):
    for column_index, (B_field, coordinates_X, B_X, coordinates_Y, B_Y, coordinates_Z, B_Z, colorbar_label, title, plot_coordinates) in enumerate(fields):
        ax = fig.add_subplot(3, 3, row_index * 3 + column_index + 1)

        if row_index == 0:
            plane_coordinates, plane_field = coordinates_X, B_X
        elif row_index == 1:
            plane_coordinates, plane_field = coordinates_Y, B_Y
        else:
            plane_coordinates, plane_field = coordinates_Z, B_Z

        x_plot = plane_coordinates[:, coordinate_1]
        y_plot = plane_coordinates[:, coordinate_2]
        u = plane_field[:, coordinate_1]
        v = plane_field[:, coordinate_2]

        quiver = ax.quiver(
            x_plot, y_plot, u, v, np.linalg.norm(plane_field, axis=1),
            cmap='viridis', scale=0.5 * 10**(-9), scale_units='width', width=0.005
        )

        xmin, xmax = x_plot.min() - 0.2, x_plot.max() + 0.2
        ymin, ymax = y_plot.min() - 0.2, y_plot.max() + 0.2
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        fig.colorbar(quiver, ax=ax, label=colorbar_label)
        ax.set_title(f'{title}: {plane_name}', pad=10)
        ax.set_xlabel(axis_1_label)
        ax.set_ylabel(axis_2_label)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)

# Formatting: increase row spacing and bring the main heading closer to the first row.
fig.subplots_adjust(hspace=0.55, top=0.93)
plt.tight_layout()
plt.show()

rms_tobi_pT = np.sqrt((1 / len(B_residual_field_tobi)) * np.sum((np.linalg.norm(B_residual_field_tobi) * 10**(12))**2))
rms_gregor_pT = np.sqrt((1 / len(B_residual_field_gregor)) * np.sum((np.linalg.norm(B_residual_field_gregor) * 10**(12))**2))
rms_difference_pT = np.sqrt((1 / len(B_difference_field)) * np.sum((np.linalg.norm(B_difference_field) * 10**(12))**2))
print(f'RMS magnitude of the Tobi residual field:               RMS = {rms_tobi_pT:.2f} pT')
print(f'RMS magnitude of the Gregor residual field:             RMS = {rms_gregor_pT:.2f} pT')
print(f'RMS magnitude of the difference field (Gregor - Tobi):  RMS = {rms_difference_pT:.2f} pT')

In [ ]:
# Specify folders containing the positive and negative nulling-field point data
nulling_field_positive_folder = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Nulling_fields\Residual_field_nulled_pos_v2_2026-09-02_10-49-38\map\points"
nulling_field_negative_folder = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Nulling_fields\Nulled_field_other_magnicon_outs_off_AGAIN_2026-09-03_21-53-03\map\points"

# If the .npz file does not contain geometric track data, specify the coordinate shifts here
coordinate_shift_x = 0.000                              # shift of the mapped volume in the x-direction
coordinate_shift_y = 0.000                              # shift of the mapped volume in the y-direction
coordinate_shift_z = 0.000                              # shift of the mapped volume in the z-direction

# Load the data using the load_data_from_folder function from External_functions.py.
nulling_field_coordinates, B_nulling_field_positive, B_nulling_field_negative = fkt.load_data_from_folder(
    nulling_field_positive_folder,
    nulling_field_negative_folder,
    coordinate_shift_x,
    coordinate_shift_y,
    coordinate_shift_z,
)

# Transform the magnetic-field components from the QSpin to the MSR coordinate system
apply_qspin_to_msr_transform = True
positive_nulling_field_offsets = [-21.43e-12, -24.31e-12, 72.39e-12]
negative_nulling_field_offsets = [-98.81, 25.98, 11.12] # Third measurement # [-35.09e-12, 25.61e-12, -8.29e-12] # First measurement # [-89.43e-12, 37.27e-12, 12.43e-12] # second measurement
remove_field_offset = False
if apply_qspin_to_msr_transform:
    Bx = B_nulling_field_positive[:, 0].copy()
    By = B_nulling_field_positive[:, 1].copy()
    Bz = B_nulling_field_positive[:, 2].copy()

    if remove_field_offset:
        Bx = Bx - positive_nulling_field_offsets[0]
        By = By - positive_nulling_field_offsets[1]
        Bz = Bz - positive_nulling_field_offsets[2]

    B_nulling_field_positive[:, 0] = Bx
    B_nulling_field_positive[:, 1] = -Bz
    B_nulling_field_positive[:, 2] = -By

    Bx = B_nulling_field_negative[:, 0].copy()
    By = B_nulling_field_negative[:, 1].copy()
    Bz = B_nulling_field_negative[:, 2].copy()

    if remove_field_offset:
        Bx = Bx - negative_nulling_field_offsets[0]
        By = By - negative_nulling_field_offsets[1]
        Bz = Bz - negative_nulling_field_offsets[2]

    B_nulling_field_negative[:, 0] = Bx
    B_nulling_field_negative[:, 1] = -Bz
    B_nulling_field_negative[:, 2] = -By
    print('Changed the magnetic-field directions from the left-handed QSpin coordinate system to the right-handed MSR coordinate system to match the spatial coordinates.')

In [ ]:
B_nulled_field_positive = B_nulling_field_positive - B_difference_field

# Plot the positive nulling field, target difference field, and resulting nulled field
fig = plt.figure(figsize=(18, 5))

fields = [
    (B_nulling_field_positive, r'$|\mathbf{B}_{\mathrm{+nulling}}|$', 'Positive Nulling Field', nulling_field_coordinates),
    (B_difference_field, r'$|\mathbf{B}_{\mathrm{difference}}|$', 'Target Difference Field', nulling_field_coordinates),
    (B_nulled_field_positive, r'$|\mathbf{B}_{\mathrm{nulled,+}}|$', 'Nulled Field (Positive)', nulling_field_coordinates),
]

for plot_index, (B_field, colorbar_label, title, plot_coordinates) in enumerate(fields):
    ax = fig.add_subplot(1, 3, plot_index + 1, projection='3d')

    field_magnitude = np.linalg.norm(B_field, axis=1)

    # Formatting: slightly reduce the scatter-marker size.
    scatter = ax.scatter(
        plot_coordinates[:, 0],
        plot_coordinates[:, 1],
        plot_coordinates[:, 2],
        c=field_magnitude,
        s=70,
        cmap='viridis',
        alpha=0.8
    )

    ax.set_title(title, pad=10)
    # Formatting: shorten and separate the 3D colourbar from the z-axis label.
    fig.colorbar(scatter, ax=ax, label=colorbar_label, pad=0.15, shrink=0.5)
    ax.set_xlabel(r'$x$ [m]')
    ax.set_ylabel(r'$y$ [m]')
    ax.set_zlabel(r'$z$ [m]')
    ax.view_init(elev=20, azim=-60)

plt.tight_layout()

# Extract the points and field vectors on each symmetry plane
coordinates_X0, B_positive_X0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulling_field_positive, 0)
coordinates_Y0, B_positive_Y0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulling_field_positive, 1)
coordinates_Z0, B_positive_Z0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulling_field_positive, 2)

coordinates_X0_difference, B_difference_X0, _ = filter_symmetry_plane(nulling_field_coordinates, B_difference_field, 0)
coordinates_Y0_difference, B_difference_Y0, _ = filter_symmetry_plane(nulling_field_coordinates, B_difference_field, 1)
coordinates_Z0_difference, B_difference_Z0, _ = filter_symmetry_plane(nulling_field_coordinates, B_difference_field, 2)

coordinates_X0_nulled, B_nulled_X0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulled_field_positive, 0)
coordinates_Y0_nulled, B_nulled_Y0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulled_field_positive, 1)
coordinates_Z0_nulled, B_nulled_Z0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulled_field_positive, 2)

# Create a 3x3 grid of vector plots for the three fields and three symmetry planes
fig = plt.figure(figsize=(18, 12))

fields = [
    (B_nulling_field_positive, coordinates_X0, B_positive_X0, coordinates_Y0, B_positive_Y0, coordinates_Z0, B_positive_Z0,
     r'$|\mathbf{B}_{\mathrm{+nulling}}|$', 'Positive Nulling Field', nulling_field_coordinates),
    (B_difference_field, coordinates_X0_difference, B_difference_X0, coordinates_Y0_difference, B_difference_Y0, coordinates_Z0_difference, B_difference_Z0,
     r'$|\mathbf{B}_{\mathrm{difference}}|$', 'Target Difference Field', nulling_field_coordinates),
    (B_nulled_field_positive, coordinates_X0_nulled, B_nulled_X0, coordinates_Y0_nulled, B_nulled_Y0, coordinates_Z0_nulled, B_nulled_Z0,
     r'$|\mathbf{B}_{\mathrm{nulled,+}}|$', 'Nulled Field (Positive)', nulling_field_coordinates),
]

symmetry_planes = [
    (r'$x$ = 0', 1, 2, r'$y$ [m]', r'$z$ [m]'),
    (r'$y$ = 0', 0, 2, r'$x$ [m]', r'$z$ [m]'),
    (r'$z$ = 0', 0, 1, r'$x$ [m]', r'$y$ [m]')
]

for row_index, (plane_name, coordinate_1, coordinate_2, axis_1_label, axis_2_label) in enumerate(symmetry_planes):
    for column_index, (B_field, coordinates_X, B_X, coordinates_Y, B_Y, coordinates_Z, B_Z, colorbar_label, title, plot_coordinates) in enumerate(fields):
        ax = fig.add_subplot(3, 3, row_index * 3 + column_index + 1)

        if row_index == 0:
            plane_coordinates, plane_field = coordinates_X, B_X
        elif row_index == 1:
            plane_coordinates, plane_field = coordinates_Y, B_Y
        else:
            plane_coordinates, plane_field = coordinates_Z, B_Z

        x_plot = plane_coordinates[:, coordinate_1]
        y_plot = plane_coordinates[:, coordinate_2]
        u = plane_field[:, coordinate_1]
        v = plane_field[:, coordinate_2]

        quiver = ax.quiver(
            x_plot, y_plot, u, v, np.linalg.norm(plane_field, axis=1),
            cmap='viridis', scale=0.5 * 10**(-9), scale_units='width', width=0.005
        )

        xmin, xmax = x_plot.min() - 0.2, x_plot.max() + 0.2
        ymin, ymax = y_plot.min() - 0.2, y_plot.max() + 0.2
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        fig.colorbar(quiver, ax=ax, label=colorbar_label)
        ax.set_title(f'{title}: {plane_name}', pad=10)
        ax.set_xlabel(axis_1_label)
        ax.set_ylabel(axis_2_label)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)

# Formatting: increase row spacing and bring the main heading closer to the first row.
fig.subplots_adjust(hspace=0.55, top=0.93)
plt.tight_layout()
plt.show()

rms_positive_nulling_pT = np.sqrt((1 / len(B_nulling_field_positive)) * np.sum((np.linalg.norm(B_nulling_field_positive) * 10**(12))**2))
rms_difference_pT = np.sqrt((1 / len(B_difference_field)) * np.sum((np.linalg.norm(B_difference_field) * 10**(12))**2))
rms_positive_nulled_pT = np.sqrt((1 / len(B_nulled_field_positive)) * np.sum((np.linalg.norm(B_nulled_field_positive) * 10**(12))**2))
print(f'RMS magnitude of the positive nulling field:    RMS = {rms_positive_nulling_pT:.2f} pT')
print(f'RMS magnitude of the target difference field:   RMS = {rms_difference_pT:.2f} pT')
print(f'RMS magnitude of the positively nulled field:   RMS = {rms_positive_nulled_pT:.2f} pT')

In [ ]:
B_nulled_field_negative = B_nulling_field_negative - B_difference_field

# Plot the negative nulling field, target difference field, and resulting nulled field
fig = plt.figure(figsize=(18, 5))

fields = [
    (B_nulling_field_negative, r'$|\mathbf{B}_{\mathrm{-nulling}}|$', 'Negative Nulling Field', nulling_field_coordinates),
    (B_difference_field, r'$|\mathbf{B}_{\mathrm{difference}}|$', 'Target Difference Field', nulling_field_coordinates),
    (B_nulled_field_negative, r'$|\mathbf{B}_{\mathrm{nulled,-}}|$', 'Nulled Field (Negative)', nulling_field_coordinates),
]

for plot_index, (B_field, colorbar_label, title, plot_coordinates) in enumerate(fields):
    ax = fig.add_subplot(1, 3, plot_index + 1, projection='3d')

    field_magnitude = np.linalg.norm(B_field, axis=1)

    # Formatting: slightly reduce the scatter-marker size.
    scatter = ax.scatter(
        plot_coordinates[:, 0],
        plot_coordinates[:, 1],
        plot_coordinates[:, 2],
        c=field_magnitude,
        s=70,
        cmap='viridis',
        alpha=0.8
    )

    ax.set_title(title, pad=10)
    # Formatting: shorten and separate the 3D colourbar from the z-axis label.
    fig.colorbar(scatter, ax=ax, label=colorbar_label, pad=0.15, shrink=0.5)
    ax.set_xlabel(r'$x$ [m]')
    ax.set_ylabel(r'$y$ [m]')
    ax.set_zlabel(r'$z$ [m]')
    ax.view_init(elev=20, azim=-60)

plt.tight_layout()

# Extract the points and field vectors on each symmetry plane
coordinates_X0, B_negative_X0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulling_field_negative, 0)
coordinates_Y0, B_negative_Y0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulling_field_negative, 1)
coordinates_Z0, B_negative_Z0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulling_field_negative, 2)

coordinates_X0_difference, B_difference_X0, _ = filter_symmetry_plane(nulling_field_coordinates, B_difference_field, 0)
coordinates_Y0_difference, B_difference_Y0, _ = filter_symmetry_plane(nulling_field_coordinates, B_difference_field, 1)
coordinates_Z0_difference, B_difference_Z0, _ = filter_symmetry_plane(nulling_field_coordinates, B_difference_field, 2)

coordinates_X0_nulled, B_nulled_X0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulled_field_negative, 0)
coordinates_Y0_nulled, B_nulled_Y0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulled_field_negative, 1)
coordinates_Z0_nulled, B_nulled_Z0, _ = filter_symmetry_plane(nulling_field_coordinates, B_nulled_field_negative, 2)

# Create a 3x3 grid of vector plots for the three fields and three symmetry planes
fig = plt.figure(figsize=(18, 12))

fields = [
    (B_nulling_field_negative, coordinates_X0, B_negative_X0, coordinates_Y0, B_negative_Y0, coordinates_Z0, B_negative_Z0,
     r'$|\mathbf{B}_{\mathrm{-nulling}}|$', 'Negative Nulling Field', nulling_field_coordinates),
    (B_difference_field, coordinates_X0_difference, B_difference_X0, coordinates_Y0_difference, B_difference_Y0, coordinates_Z0_difference, B_difference_Z0,
     r'$|\mathbf{B}_{\mathrm{difference}}|$', 'Target Difference Field', nulling_field_coordinates),
    (B_nulled_field_negative, coordinates_X0_nulled, B_nulled_X0, coordinates_Y0_nulled, B_nulled_Y0, coordinates_Z0_nulled, B_nulled_Z0,
     r'$|\mathbf{B}_{\mathrm{nulled,-}}|$', 'Nulled Field (Negative)', nulling_field_coordinates),
]

symmetry_planes = [
    (r'$x$ = 0', 1, 2, r'$y$ [m]', r'$z$ [m]'),
    (r'$y$ = 0', 0, 2, r'$x$ [m]', r'$z$ [m]'),
    (r'$z$ = 0', 0, 1, r'$x$ [m]', r'$y$ [m]')
]

for row_index, (plane_name, coordinate_1, coordinate_2, axis_1_label, axis_2_label) in enumerate(symmetry_planes):
    for column_index, (B_field, coordinates_X, B_X, coordinates_Y, B_Y, coordinates_Z, B_Z, colorbar_label, title, plot_coordinates) in enumerate(fields):
        ax = fig.add_subplot(3, 3, row_index * 3 + column_index + 1)

        if row_index == 0:
            plane_coordinates, plane_field = coordinates_X, B_X
        elif row_index == 1:
            plane_coordinates, plane_field = coordinates_Y, B_Y
        else:
            plane_coordinates, plane_field = coordinates_Z, B_Z

        x_plot = plane_coordinates[:, coordinate_1]
        y_plot = plane_coordinates[:, coordinate_2]
        u = plane_field[:, coordinate_1]
        v = plane_field[:, coordinate_2]

        quiver = ax.quiver(
            x_plot, y_plot, u, v, np.linalg.norm(plane_field, axis=1),
            cmap='viridis', scale=0.5 * 10**(-9), scale_units='width', width=0.005
        )

        xmin, xmax = x_plot.min() - 0.2, x_plot.max() + 0.2
        ymin, ymax = y_plot.min() - 0.2, y_plot.max() + 0.2
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        fig.colorbar(quiver, ax=ax, label=colorbar_label)
        ax.set_title(f'{title}: {plane_name}', pad=10)
        ax.set_xlabel(axis_1_label)
        ax.set_ylabel(axis_2_label)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)

# Formatting: increase row spacing and bring the main heading closer to the first row.
fig.subplots_adjust(hspace=0.55, top=0.93)
plt.tight_layout()
plt.show()

rms_negative_nulling_pT = np.sqrt((1 / len(B_nulling_field_negative)) * np.sum((np.linalg.norm(B_nulling_field_negative) * 10**(12))**2))
rms_difference_pT = np.sqrt((1 / len(B_difference_field)) * np.sum((np.linalg.norm(B_difference_field) * 10**(12))**2))
rms_negative_nulled_pT = np.sqrt((1 / len(B_nulled_field_negative)) * np.sum((np.linalg.norm(B_nulled_field_negative) * 10**(12))**2))
print(f'RMS magnitude of the negative nulling field:    RMS = {rms_negative_nulling_pT:.2f} pT')
print(f'RMS magnitude of the target difference field:   RMS = {rms_difference_pT:.2f} pT')
print(f'RMS magnitude of the negatively nulled field:   RMS = {rms_negative_nulled_pT:.2f} pT')